<a href="https://colab.research.google.com/github/hpsj2712/atelie-generativo-individual-homerio/blob/main/notebooks/02_treino_lora_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install diffusers transformers accelerate peft datasets
!git clone https://github.com/huggingface/diffusers
%cd diffusers/examples/text_to_image
from huggingface_hub import login
from google.colab import userdata
token = userdata.get('hugging-face')

login(token)

Cloning into 'diffusers'...
remote: Enumerating objects: 126316, done.
remote: Counting objects: 100% (503/503), done.
remote: Compressing objects: 100% (238/238), done.
remote: Total 126316 (delta 377), reused 266 (delta 264), pack-reused 125813 (from 2)
Receiving objects: 100% (126316/126316), 100.35 MiB | 25.87 MiB/s, done.
Resolving deltas: 100% (94078/94078), done.
/content/diffusers/examples/text_to_image


In [2]:
# Desinstala a versão atual do diffusers instalada via pip
!pip uninstall -y diffusers

Found existing installation: diffusers 0.38.0
Uninstalling diffusers-0.38.0:
  Successfully uninstalled diffusers-0.38.0


In [3]:
# Instala o diffusers a partir do código-fonte clonado em modo editável
# Isso garantirá que a versão mais recente ou a versão do branch principal seja usada.
%cd /content/diffusers
!pip install -e .
%cd examples/text_to_image

/content/diffusers
Obtaining file:///content/diffusers
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for diffusers (pyproject.toml) ... done
  Created wheel for diffusers: filename=diffusers-0.40.0.dev0-0.editable-py3-none-any.whl size=11519 sha256=44e370194cba24147ae281a004e644071e5f9d18aa14f962cf5a783c96a19201
  Stored in directory: /tmp/pip-ephem-wheel-cache-s93nic78/wheels/8a/fc/09/385efb77b455b2fd4a656c950079c93147e1f50ae614e51beb
Successfully built diffusers
/content/diffusers/examples/text_to_image


In [4]:
from datasets import load_dataset
import pandas as pd
import os

# 1. Clone o repositório
!git clone https://github.com/hpsj2712/atelie-generativo-individual-homerio /content/temp_repo

# 2. Prepare o diretório do dataset
!mkdir -p /content/dataset

# 3. Mova as imagens para a pasta final
!mv /content/temp_repo/dados/*.jpg /content/dataset/

# 4. Processar as legendas (legendas.txt)
# O formato atual é "nome.jpg: legenda"
legendas_dict = {}
with open('/content/temp_repo/dados/legendas.txt', 'r', encoding='utf-8') as f:
    for line in f:
        if ":" in line:
            parts = line.split(":", 1)
            file_name = parts[0].strip()
            caption = parts[1].strip()
            legendas_dict[file_name] = caption

# 5. Criar o metadata.csv no formato exigido pelo Diffusers
df = pd.DataFrame(list(legendas_dict.items()), columns=['file_name', 'text'])
df.to_csv('/content/dataset/metadata.csv', index=False)

print("Dataset e metadata.csv criados com sucesso em /content/dataset/")
!rm -rf /content/temp_repo

Cloning into '/content/temp_repo'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 149 (delta 39), reused 31 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 128.40 MiB | 25.51 MiB/s, done.
Resolving deltas: 100% (39/39), done.
Dataset e metadata.csv criados com sucesso em /content/dataset/


In [7]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
!accelerate launch train_text_to_image_lora.py \
  --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
  --train_data_dir="/content/dataset" \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --max_train_steps=1000 \
  --learning_rate=3e-4 \
  --lr_scheduler="cosine" \
  --rank=5 \
  --mixed_precision="fp16" \
  --checkpointing_steps=200 \
  --validation_prompt="estilo_arquitetura_modernista_brasilia, a concret square with a museum" \
  --output_dir="/content/lora_saida" \
  --push_to_hub \
  --hub_model_id="homerio/estilo_arquitetura_modernista_brasilia_v1"

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorc